# CREDIT SCORING MODEL: PREDICTING CREDITWORTHINESS USING MACHINE LEARNING
### An End-to-End Applied Machine Learning Project

**Author / Lead ML Engineer:** Machine Learning & Financial Risk Modeling Group  
**Domain:** Financial Technology (FinTech) & Banking Underwriting  
**Dataset:** UCI Statlog German Credit Data (1,000 Authentic Records, 20 Attributes)  
**Target Variable:** Credit Risk (`Good Credit` / `Bad Credit - High Risk`)  

---
## 1. Executive Summary & Problem Formulation
Credit scoring is one of the foundational statistical applications in commercial lending and retail banking. When an applicant requests credit (personal loan, automotive financing, or mortgage), the lending institution must assess the likelihood of default (`creditworthiness`).

### The Asymmetric Cost of Credit Risk Errors:
- **False Negative (FN):** Classifying a high-risk defaulting borrower as creditworthy. This causes severe capital loss (loss of principal balance), typically estimated at **4x to 5x** the cost of a false rejection.
- **False Positive (FP):** Classifying a reliable borrower as high-risk. This incurs an opportunity cost (lost interest income and customer dissatisfaction).

Therefore, the positive target class in this study is defined as **High Credit Risk (`bad` = 1)**, prioritizing **High-Risk Recall** and **ROC-AUC** alongside overall accuracy.

## 2. Environment Setup and Library Imports

In [ ]:
import os
import sys
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project modules are resolvable
sys.path.append(os.path.abspath('..'))
from src.data_preprocessing import load_dataset, prepare_train_test_split, build_preprocessor_pipeline
from src.feature_engineering import FinancialFeatureEngineer
from src.prediction import predict_credit_risk, batch_predict

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
print('Environment and libraries initialized successfully.')

## 3. Dataset Loading, Inspection & Provenance
We utilize the authentic **Statlog German Credit Dataset** sourced from the UCI Machine Learning Repository and OpenML (ID: `credit-g`). It contains 1,000 real applicant observations across 20 financial and demographic indicators.

In [ ]:
data_path = '../data/credit_dataset.csv'
df = pd.read_csv(data_path)

print(f'Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'Missing Values: {df.isnull().sum().sum()}')
print('\nTarget Class Balance:')
print(df['class'].value_counts())
print('\nPercentage Distribution:')
print(df['class'].value_counts(normalize=True) * 100)
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's explore the key distributions, financial exposure differences, and risk drivers.

In [ ]:
# 1. Visualizing Target Distribution
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x='class', palette=['#2b5c8f', '#d95f02'])
plt.title('Credit Risk Class Imbalance (70% Good vs 30% Bad)')
plt.xlabel('Credit Risk Class')
plt.ylabel('Applicant Count')
for p in ax.patches:
    ax.annotate(f'{p.get_height()}', (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                ha='center', va='center', color='white', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 2. Credit Amount and Loan Duration Distribution by Risk
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(x='class', y='credit_amount', data=df, ax=axes[0], palette=['#4575b4', '#d73027'])
axes[0].set_title('Requested Credit Amount by Risk Class')

sns.boxplot(x='class', y='duration', data=df, ax=axes[1], palette=['#4575b4', '#d73027'])
axes[1].set_title('Loan Duration in Months by Risk Class')
plt.tight_layout()
plt.show()

In [ ]:
# 3. Checking Account Liquidity vs Default Rate
checking_risk = df.groupby('checking_status')['class'].apply(lambda s: (s == 'bad').mean() * 100).sort_values(ascending=False)
plt.figure(figsize=(8, 4))
sns.barplot(x=checking_risk.values, y=checking_risk.index, palette='Reds_r')
plt.title('High-Risk Default Rate by Checking Account Status (%)')
plt.xlabel('Bad Credit Proportion (%)')
plt.tight_layout()
plt.show()

## 5. Domain Feature Engineering
Raw financial variables can be combined into highly predictive credit indicators:
1. `monthly_payment_burden = credit_amount / duration`
2. `credit_to_age_ratio = credit_amount / age`
3. `liquidity_buffer_score = checking_status_score + savings_status_score`
4. `high_burden_flag = (installment_commitment >= 4) & (monthly_burden > 150)`
5. `stability_composite = (employment_tenure * 1.5) + residence_tenure`

In [ ]:
fe = FinancialFeatureEngineer()
df_eng = fe.fit_transform(df.drop(columns=['class']))
print('Engineered feature matrix preview:')
df_eng[['monthly_payment_burden', 'credit_to_age_ratio', 'high_burden_flag', 'liquidity_buffer_score', 'stability_composite']].head()

## 6. Preprocessing & Stratified Train-Test Splitting
We apply a strict **80/20 stratified split** ensuring zero data leakage. All imputations, scalers, and one-hot encoders are fitted strictly on the training partition.

In [ ]:
X_train, X_test, y_train, y_test = prepare_train_test_split(df, test_size=0.2, random_state=42)
print(f'Training Samples: {len(X_train)} (Bad Credit Count: {y_train.sum()})')
print(f'Test Samples:     {len(X_test)} (Bad Credit Count: {y_test.sum()})')

preprocessor = build_preprocessor_pipeline()
X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)
print(f'Feature Matrix Dimensions: {X_train_trans.shape[1]} columns')

## 7. Machine Learning Algorithms Benchmark
We train and tune three distinct machine learning paradigms:
1. **Logistic Regression (Linear Regularized Model)**
2. **Decision Tree Classifier (Non-linear Rule-based Model)**
3. **Random Forest Classifier (Ensemble Bagging Model)**

In [ ]:
# Load pre-trained, tuned model pipelines
lr_pipe = joblib.load('../models/logistic_regression_model.joblib')
dt_pipe = joblib.load('../models/decision_tree_model.joblib')
rf_pipe = joblib.load('../models/random_forest_model.joblib')

comparison_df = pd.read_csv('../outputs/evaluation_results/model_comparison.csv')
print('Test Set Performance Benchmark:')
display(comparison_df[['Model', 'Accuracy', 'ROC-AUC', 'High-Risk Recall', 'High-Risk Precision', 'High-Risk F1-Score', 'Normalized Risk Cost']])

## 8. Model Evaluation Visualizations
Let's display the generated ROC Curves, Confusion Matrices, and Random Forest Feature Importances.

In [ ]:
from IPython.display import Image, display

print('ROC Curves:')
display(Image('../outputs/graphs/08_roc_curves.png'))

print('Confusion Matrices (TP, FP, TN, FN):')
display(Image('../outputs/graphs/07_confusion_matrices.png'))

print('Top 15 Random Forest Feature Importances:')
display(Image('../outputs/graphs/11_feature_importance_rf.png'))

## 9. Real-Time Inference & Explainability Function
Let's run a live creditworthiness prediction on two distinct applicant profiles.

In [ ]:
sample_high_risk = {
    'checking_status': '<0',
    'duration': 48,
    'credit_history': 'delayed previously',
    'purpose': 'business',
    'credit_amount': 9500,
    'savings_status': '<100',
    'employment': '<1',
    'installment_commitment': 4,
    'personal_status': 'male single',
    'other_parties': 'none',
    'residence_since': 1,
    'property_magnitude': 'no known property',
    'age': 22,
    'other_payment_plans': 'bank',
    'housing': 'rent',
    'existing_credits': 2,
    'job': 'unskilled resident',
    'num_dependents': 1,
    'own_telephone': 'none',
    'foreign_worker': 'yes'
}

prediction = predict_credit_risk(sample_high_risk, model_path='../models/best_credit_model.joblib')
print(f"Predicted Class:      {prediction['predicted_risk_label']}")
print(f"High-Risk Probability: {prediction['high_risk_probability']:.2%}")
print(f"Model Confidence:      {prediction['model_confidence']:.2%}")
print('\nKey Explanatory Risk Factors:')
for factor in prediction['key_risk_factors']:
    print(f" - {factor}")
print(f"\n{prediction['limitations_disclaimer']}")

## 10. Fairness, Ethics, and Regulatory Constraints
In production credit decisioning systems, strict regulatory standards apply:
1. **Disparate Impact & Protected Classes:** Features such as `personal_status` (encoding sex and marital status), `foreign_worker`, and `age` must be audited for disparate impact under the Equal Credit Opportunity Act (ECOA) and EU Artificial Intelligence Act.
2. **Adverse Action Transparency:** Applicants denied credit have a legal right to adverse action notices detailing the specific factors that contributed to rejection.
3. **Human-in-the-Loop Governance:** Automated classification should serve as underwriting decision support rather than an unchecked automated rejection engine.